In [1]:
import xarray as xr
import torch
import sys
sys.path.append('../src')
from constants import INPT_VARS, EXTRA_VARS, OUT_VARS
import os
import numpy as np
from einops import rearrange

In [2]:
data_dir = "/pscratch/sd/s/suryad/data"
wet_file = "3D_wet.pt"
surface_wet_file = "surface_wet.pt"
data_zarr = "3D_data"
data_means_zarr = "3D_data_means"
data_stds_zarr = "3D_data_stds"
grid_file = "Grid_New.nc"

inputs = INPT_VARS["3D_all"]
extra_in = EXTRA_VARS["3D_all"]
outputs = OUT_VARS["3D_all"]
lag = 1
interval = 1
hist = 0
N_samples = 20
N_val = 10
steps = 4


wet = torch.load(
    os.path.join(data_dir, wet_file)
)
data = xr.open_zarr(
    os.path.join(data_dir, data_zarr)
)
data_mean = xr.open_zarr(
    os.path.join(data_dir, data_means_zarr)
)
data_std = xr.open_zarr(
    os.path.join(data_dir, data_stds_zarr)
)
    


### Data Loading

#### data_CNN_Disk

#### old

In [3]:
class old_data_CNN_Disk(torch.utils.data.Dataset):

    def __init__(
        self,
        data,
        inputs_str,
        extra_in_str,
        outputs_str,
        wet,
        data_mean,
        data_std,
        n_samples,
        lag,
        interval,
        ind_start,
        device="cuda",
    ):
        super().__init__()
        self.device = device

        self.size = n_samples
        self.lag = lag
        self.interval = interval
        self.ind_start = ind_start

        self.inputs = data[inputs_str + extra_in_str]
        self.outputs = data[outputs_str]

        self.in_mean = data_mean[inputs_str + extra_in_str]
        self.in_std = data_std[inputs_str + extra_in_str]

        self.out_mean = data_mean[outputs_str]
        self.out_std = data_std[outputs_str]

        self.wet = wet

    def set_device(self, device):
        self.device = device

    def __len__(self):
        # Number of data point we have. Alternatively self.data.shape[0], or self.label.shape[0]
        return self.size

    def __getitem__(self, idx):
        # Return the idx-th data point of the dataset
        # If we have multiple things to return (data point and label), we can return them as tuple
        if type(idx) == list:
            ind_in = [self.ind_start + i * self.interval for i in idx]
            ind_out = [self.ind_start + i * self.interval + self.lag for i in idx]
        elif type(idx) == slice:
            if idx.start == None and idx.stop == None:
                idx = slice(0, self.size, idx.step)
            elif idx.start == None:
                idx = slice(0, idx.stop, idx.step)
            elif idx.stop == None:
                idx = slice(idx.start, self.size, idx.step)

            ind_in = slice(
                self.ind_start + idx.start, self.ind_start + idx.stop * self.interval, self.interval
            )
            ind_out = slice(
                self.ind_start + idx.start + self.lag,
                self.ind_start + idx.stop * self.interval + self.lag,
                self.interval,
            )
        if type(idx) == int:
            ind_in = self.ind_start + idx * self.interval
            ind_out = self.ind_start + idx * self.interval + self.lag

        data_in = self.inputs.isel(time=ind_in)
        data_in = ((data_in - self.in_mean) / self.in_std).fillna(0)
        label = self.outputs.isel(time=ind_out)
        label = ((label - self.out_mean) / self.out_std).fillna(0)

        if type(idx) == int:
            data_in = data_in.to_array().transpose("variable", "y", "x").to_numpy()
            label = label.to_array().transpose("variable", "y", "x").to_numpy()
        else:
            data_in = (
                data_in.to_array().transpose("time", "variable", "y", "x").to_numpy()
            )
            label = label.to_array().transpose("time", "variable", "y", "x").to_numpy()

        items = (torch.from_numpy(data_in).float(), torch.from_numpy(label).float())

        return items

#### new

In [37]:
import xarray as xr
import numpy as np
import torch
import torch.nn as nn
import torch.utils.data as data
from scipy.ndimage import gaussian_filter
from einops import rearrange
import os


class data_CNN_Disk(torch.utils.data.Dataset):

    def __init__(
        self,
        data,
        inputs_str,
        extra_in_str,
        outputs_str,
        wet,
        data_mean,
        data_std,
        n_samples,
        lag,
        interval,
        hist,
        ind_start,
        long_rollout,
        device="cuda",
    ):
        super().__init__()
        self.device = device

        self.size = n_samples
        self.lag = lag
        self.interval = interval
        self.hist = hist
        self.ind_start = ind_start
        
        assert self.interval == 1
        assert self.lag == 1
        
        data = data.isel(time=slice(self.ind_start, None))
        self.inputs = data[inputs_str + extra_in_str]
        self.outputs = data[outputs_str]
        self.inputs_no_extra = data[inputs_str]
        self.extras = data[extra_in_str]

        # Rolling indices to keep track of histories/past states
        # We would want to return a tuple of input and output data like this:
        # HIST=0 ; 0->[0, 1]; 1->[1, 2]; 2->[2, 3]; 3->[3, 4]
        # HIST=1 ; 0->[[0, 1], [2, 3]]; 1->[[1, 2], [3, 4]]; 2->[[2, 3], [4, 5]]; 3->[[3, 4], [5, 6]]
        # HIST=2 ; 0->[[0, 1, 2], [3, 4, 5]]; 1->[[1, 2, 3], [4, 5, 6]]; 2->[[2, 3, 4], [5, 6, 7]]; 3->[[3, 4, 5], [6, 7, 8]]
        indices = xr.DataArray(
            np.arange(data.time.size),
            dims=["time"],
            coords={"time": data.time},
        )
        total_steps = 2*self.hist + 1
        rolling_indices = indices.rolling(time=len(data.time)-total_steps, center=False).construct("window_dim").astype(int)
        rolling_indices = rolling_indices.transpose('window_dim', 'time').isel(time=slice(len(data.time)-total_steps-1, None)) # Remove first few null indices
        self.rolling_indices = rolling_indices.isel(window_dim=slice(0, None, self.hist+1))
        
        if long_rollout:
            window0 = self.rolling_indices.isel(window_dim=0)
            print("Long rollout will begin with input and produce output from time index {0} and {1} respectively".format(window0.isel(time=0).values+ind_start, window0.isel(time=self.hist+1).values+ind_start))

        self.in_mean = data_mean[inputs_str + extra_in_str]
        self.in_std = data_std[inputs_str + extra_in_str]
        self.out_mean = data_mean[outputs_str]
        self.out_std = data_std[outputs_str]
        self.inputs_no_extra_mean = data_mean[inputs_str]
        self.inputs_no_extra_std = data_std[inputs_str]
        self.extras_mean = data_mean[extra_in_str]
        self.extras_std = data_std[extra_in_str]

        self.wet = wet

    def set_device(self, device):
        self.device = device

    def __len__(self):
        return self.rolling_indices.window_dim.size

    def __getitem__(self, idx):
        if type(idx) == slice:
            if idx.start == None and idx.stop == None:
                idx = slice(0, self.size, idx.step)
            elif idx.start == None:
                idx = slice(0, idx.stop, idx.step)
            elif idx.stop == None:
                idx = slice(idx.start, self.size, idx.step)
        elif type(idx) == int:
            idx = slice(idx, idx + 1, 1)
        
        x_index = xr.Variable(["window_dim", "time"], self.rolling_indices.isel(window_dim=idx))
        data_in = self.inputs_no_extra.isel(time=x_index).isel(time=slice(None, self.hist+1))
        data_in = ((data_in - self.inputs_no_extra_mean) / self.inputs_no_extra_std).fillna(0)
        data_in = data_in.to_array().transpose("window_dim", "time", "variable", "y", "x").to_numpy()
        data_in = rearrange(data_in, "window_dim time variable y x -> window_dim (time variable) y x")
        data_in_boundary = self.extras.isel(time=x_index).isel(time=self.hist)
        data_in_boundary = ((data_in_boundary - self.extras_mean) / self.extras_std).fillna(0)
        data_in_boundary = data_in_boundary.to_array().transpose("window_dim", "variable", "y", "x").to_numpy()
        data_in = np.concatenate((data_in, data_in_boundary), axis=1)

        label = self.outputs.isel(time=x_index).isel(time=slice(self.hist+1, None))
        label = ((label - self.out_mean) / self.out_std).fillna(0)
        label = label.to_array().transpose("window_dim", "time", "variable", "y", "x").to_numpy()
        label = rearrange(label, "window_dim time variable y x -> window_dim (time variable) y x")

        items = (torch.from_numpy(data_in).float(), torch.from_numpy(label).float())

        return items

In [33]:
hist = 0
s_train = lag * hist
e_train = s_train + N_samples * interval
e_test = e_train + interval * N_val
data = xr.open_zarr(
    os.path.join(data_dir, data_zarr)
)
val_data = data_CNN_Disk(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_val,
    lag,
    interval,
    hist,
    e_train,
    long_rollout=True,
    device="cuda",
)
# v, x_index = val_data[0]
# print(x_index)

# v, x_index = val_data[1]
# print(x_index)

/global/common/software/m4331/suryad/conda/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


Long rollout will begin with input and produce output from time index 20 and 21 respectively
<xarray.Variable (window_dim: 1, time: 2)>
array([[20, 21]])
<xarray.Variable (window_dim: 1, time: 2)>
array([[21, 22]])


In [14]:
assert v[0].shape[1] == (v[1].shape[1] + 3), f"Input shape {v[0].shape} and output shape {v[1].shape} do not match"

In [70]:
hist = 0 
val_data = data_CNN_Disk(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_val,
    lag,
    interval,
    hist,
    e_train,
    long_rollout=False,
    device="cuda",
)

old_val_data = old_data_CNN_Disk(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_val,
    lag,
    interval,
    e_train,
    device="cuda",
)

/global/common/software/m4331/suryad/conda/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


In [71]:
assert (old_val_data[:][1] == val_data[:][1]).all()
assert (old_val_data[:][0] == val_data[:][0]).all()

In [72]:
%%time
d0 = val_data[6]
print(d0[0].shape)
print(d0[1].shape)

torch.Size([1, 80, 180, 360])
torch.Size([1, 77, 180, 360])
CPU times: user 3.7 s, sys: 418 ms, total: 4.12 s
Wall time: 2.6 s


In [73]:
%%time
s = slice(6, 10)
d1 = val_data[s]
print(d1[0].shape)
print(d1[1].shape)

torch.Size([4, 80, 180, 360])
torch.Size([4, 77, 180, 360])
CPU times: user 4.95 s, sys: 1.52 s, total: 6.47 s
Wall time: 3.52 s


In [74]:
# Assertion
idx = s
ind_out = slice(
    idx.start + val_data.hist + val_data.lag,
    idx.stop * val_data.interval + val_data.hist + val_data.lag,
    val_data.interval,
)
assert np.nansum(val_data.outputs.isel(time=ind_out).to_array().to_numpy() - val_data.outputs.isel(time=xr.Variable(["window_dim", "time"], val_data.rolling_indices.isel(window_dim=idx))).isel(time=-1).to_array().to_numpy()) == 0.0

In [75]:
assert (d1[0][2][:77] == d1[1][1]).all()
assert (d0[1] == d1[1][0]).all()

In [76]:
hist = 1
val_data = data_CNN_Disk(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_val,
    lag,
    interval,
    hist,
    e_train,
    long_rollout=False,
    device="cuda",
)

/global/common/software/m4331/suryad/conda/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


In [77]:
%%time
d2 = val_data[6:10]
print(d2[0].shape)
print(d2[1].shape)

torch.Size([4, 157, 180, 360])
torch.Size([4, 154, 180, 360])
CPU times: user 6.08 s, sys: 2.51 s, total: 8.59 s
Wall time: 3.84 s


In [78]:
# HIST=0 ; 0->[0, 1]; 1->[1, 2]; 2->[2, 3]; 3->[3, 4]
# HIST=1 ; 0->[[0, 1], [2, 3]]; 1->[[1, 2], [3, 4]]; 2->[[2, 3], [4, 5]]; 3->[[3, 4], [5, 6]]
    
assert (d2[0][0][:77]==d1[0][0][:77]).all()
assert (d2[0][0][77:154]==d1[1][0]).all()
assert (d2[1][0][:77]==d1[1][1]).all()
assert (d2[1][0][77:154]==d1[1][2]).all()

#### data_CNN_Disk_steps

##### old

In [79]:
class old_data_CNN_Disk_steps(torch.utils.data.Dataset):

    def __init__(
        self,
        data,
        inputs_str,
        extra_in_str,
        outputs_str,
        wet,
        data_mean,
        data_std,
        n_samples,
        lag,
        interval,
        steps,
        device="cuda",
    ):
        super().__init__()
        self.device = device

        self.size = n_samples
        self.lag = lag
        self.interval = interval
        self.steps = steps
        
        self.inputs = data[inputs_str + extra_in_str]
        self.outputs = data[outputs_str]

        self.in_mean = data_mean[inputs_str + extra_in_str]
        self.in_std = data_std[inputs_str + extra_in_str]

        self.out_mean = data_mean[outputs_str]
        self.out_std = data_std[outputs_str]

        self.wet = wet

    def set_device(self, device):
        self.device = device

    def __len__(self):
        # Number of data point we have. Alternatively self.data.shape[0], or self.label.shape[0]
        return self.size

    def __getitem__(self, idx):
        # Return the idx-th data point of the dataset
        # If we have multiple things to return (data point and label), we can return them as tuple
        outputs = []
        for step in range(self.steps):
            if type(idx) == list:
                ind_in = [i * self.interval + self.lag * step for i in idx]
                ind_out = [i * self.interval + self.lag * (step + 1) for i in idx]

            elif type(idx) == slice:
                if idx.start == None and idx.stop == None:
                    idx = slice(0, self.size, idx.step)
                elif idx.start == None:
                    idx = slice(0, idx.stop, idx.step)
                elif idx.stop == None:
                    idx = slice(idx.start, self.size, idx.step)

                ind_in = slice(
                    idx.start, idx.stop * self.interval + self.lag * step, self.interval
                )
                ind_out = slice(
                    idx.start + self.lag,
                    idx.stop * self.interval + self.lag * (step + 1),
                    self.interval,
                )

            if type(idx) == int:
                ind_in = idx * self.interval + self.lag * step
                ind_out = idx * self.interval + self.lag * (step + 1)

            data_in = self.inputs.isel(time=ind_in)
            data_in = ((data_in - self.in_mean) / self.in_std).fillna(0)
            label = self.outputs.isel(time=ind_out)
            label = ((label - self.out_mean) / self.out_std).fillna(0)

            if type(idx) == int:
                data_in = data_in.to_array().transpose("variable", "y", "x").to_numpy()
                label = label.to_array().transpose("variable", "y", "x").to_numpy()
            else:
                data_in = (
                    data_in.to_array()
                    .transpose("time", "variable", "y", "x")
                    .to_numpy()
                )
                label = (
                    label.to_array().transpose("time", "variable", "y", "x").to_numpy()
                )

            outputs.append(torch.from_numpy(data_in).float())
            outputs.append(torch.from_numpy(label).float())

        return outputs

##### new

In [3]:

class data_CNN_Disk_steps(torch.utils.data.Dataset):

    def __init__(
        self,
        data,
        inputs_str,
        extra_in_str,
        outputs_str,
        wet,
        data_mean,
        data_std,
        n_samples,
        lag,
        interval,
        hist,
        steps,
        device="cuda",
    ):
        super().__init__()
        self.device = device

        self.size = n_samples
        self.lag = lag
        self.interval = interval
        self.hist = hist
        self.steps = steps
        
        assert self.interval == 1
        assert self.lag == 1
        
        self.inputs = data[inputs_str + extra_in_str]
        self.outputs = data[outputs_str]
        self.inputs_no_extra = data[inputs_str]
        self.extras = data[extra_in_str]
        
        # Rolling indices to keep track of histories/past states
        # We would want to return a tuple of input and output data like this (interval=lag=1 for now):
        # HIST=0, 4 steps ; 0->[0in, 1out, 1in, 2out, 2in, 3out, 3in, 4out]
        # HIST=1, 4 steps , 0->[[0in, 1in], [2out, 3out], [1in, 2in], [3out, 4out], [2in, 3in], [4out, 5out], [3in, 4in], [5out, 6out]]
        # HIST=2, 4 steps , 0->[[0in, 1in, 2in], [3out, 4out, 5out], [1in, 2in, 3in], [4out, 5out, 6out], [2in, 3in, 4in], [5out, 6out, 7out], [3in, 4in, 5in], [6out, 7out, 8out]]
        indices = xr.DataArray(
            np.arange(data.time.size),
            dims=["time"],
            coords={"time": data.time},
        )
        total_steps = 2*self.hist + 1
        rolling_indices = indices.rolling(time=len(data.time)-total_steps, center=False).construct("window_dim").astype(int)
        self.rolling_indices = rolling_indices.transpose('window_dim', 'time').isel(time=slice(len(data.time)-total_steps-1, None)) # Remove first few null indices
        
        self.inputs_no_extra_mean = data_mean[inputs_str]
        self.inputs_no_extra_std = data_std[inputs_str]
        self.extras_mean = data_mean[extra_in_str]
        self.extras_std = data_std[extra_in_str]
        self.in_mean = data_mean[inputs_str + extra_in_str]
        self.in_std = data_std[inputs_str + extra_in_str]

        self.out_mean = data_mean[outputs_str]
        self.out_std = data_std[outputs_str]

        self.wet = wet

    def set_device(self, device):
        self.device = device

    def __len__(self):
        return self.rolling_indices.window_dim.size - self.steps
    
    def __getitem__(self, idx):
        outputs = []
        
        assert type(idx) == int
        for step in range(self.steps): 
            
            start = idx + step
            end = idx + step + 1
            idx_slice = slice(
                    start, end, self.interval
            ) # Create a slice for similar indexing as in data_CNN_Disk
            x_index = xr.Variable(["window_dim", "time"], self.rolling_indices.isel(window_dim=idx_slice))
            data_in = self.inputs_no_extra.isel(time=x_index).isel(time=slice(None, self.hist+1))
            data_in = ((data_in - self.inputs_no_extra_mean) / self.inputs_no_extra_std).fillna(0)
            data_in = data_in.to_array().transpose("window_dim", "time", "variable", "y", "x").to_numpy()
            data_in = rearrange(data_in, "window_dim time variable y x -> window_dim (time variable) y x")
            data_in_boundary = self.extras.isel(time=x_index).isel(time=self.hist)
            data_in_boundary = ((data_in_boundary - self.extras_mean) / self.extras_std).fillna(0)
            data_in_boundary = data_in_boundary.to_array().transpose("window_dim", "variable", "y", "x").to_numpy()
            data_in = np.concatenate((data_in, data_in_boundary), axis=1).squeeze()

            label = self.outputs.isel(time=x_index).isel(time=slice(self.hist+1, None))
            label = ((label - self.out_mean) / self.out_std).fillna(0)
            label = label.to_array().transpose("window_dim", "time", "variable", "y", "x").to_numpy()
            label = rearrange(label, "window_dim time variable y x -> window_dim (time variable) y x").squeeze()

            outputs.append(torch.from_numpy(data_in).float())
            outputs.append(torch.from_numpy(label).float())

        return outputs


In [128]:
train_data = old_data_CNN_Disk_steps(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_samples,
    lag,
    interval,
    steps,
    device="cuda",
)

d0 = train_data[2]
print(d0[0].shape)
print(d0[1].shape)

torch.Size([80, 180, 360])
torch.Size([77, 180, 360])


In [129]:
hist = 0
train_data = data_CNN_Disk_steps(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_samples,
    lag,
    interval,
    hist,
    steps,
    device="cuda",
)

d = train_data[2]
print(d[0].shape)
print(d[1].shape)

/global/common/software/m4331/suryad/conda/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


torch.Size([80, 180, 360])
torch.Size([77, 180, 360])


In [130]:
assert (d[0] == d0[0]).all()
assert (d[1] == d0[1]).all()

In [131]:
for i in range(len(d0)):
    if i % 2 == 0:
        assert (d0[i][:77] == d[i][:77]).all()
    else:
        assert (d0[i] == d[i]).all()

In [132]:
hist = 1
train_data = data_CNN_Disk_steps(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_samples,
    lag,
    interval,
    hist,
    steps,
    device="cuda",
)


/global/common/software/m4331/suryad/conda/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


In [133]:
d2 = train_data[2]
print(d2[0].shape)
print(d2[1].shape)

torch.Size([157, 180, 360])
torch.Size([154, 180, 360])


In [136]:
# HIST=0, 4 steps ; 0->[0in, 1out, 1in, 2out, 2in, 3out, 3in, 4out]
# HIST=1, 4 steps , 0->[[0in, 1in], [2out, 3out], [1in, 2in], [3out, 4out], [2in, 3in], [4out, 5out], [3in, 4in], [5out, 6out]]
# HIST=2, 4 steps , 0->[[0in, 1in, 2in], [3out, 4out, 5out], [1in, 2in, 3in], [4out, 5out, 6out], [2in, 3in, 4in], [5out, 6out, 7out], [3in, 4in, 5in], [6out, 7out, 8out]]
print(d2[1].shape)
assert (d0[0][:77] == d2[0][:77]).all()
assert (d0[1] == d2[0][77:154]).all()
assert (d0[3] == d2[1][:77]).all()
assert (d0[5] == d2[1][77:]).all()
assert (d0[7] == d2[3][77:]).all()

torch.Size([154, 180, 360])


### Model Forward/Inference

In [28]:

def forward_once(input_tensor):
    return input_tensor[:, :output_channels]


def forward_once(input_tensor):
    return input_tensor[:, :output_channels]

def forward(
        inputs,
        output_only_last=False,
        loss_fn=None,
    ) -> torch.Tensor:
        outputs = []
        loss = None
        
        ### TEMP
        for n in range(len(inputs)):
            inputs[n] = inputs[n].unsqueeze(0)
        
        N, C, H, W = inputs[0].shape

        for step in range(len(inputs) // 2):
            print(step)
            if step == 0:
                input_tensor = inputs[0]
            elif step <= hist:
                inputs_0 = inputs[2 * step][:, :output_channels // (hist+1) * (hist-step+1)]
                inputs_1 = outputs[0][:, output_channels // (hist+1)  * (hist-step+1) :]
                print("inputs_0", inputs_0.shape, "slice: ", None, output_channels// (hist+1) *(hist-step+1))
                print("inputs_1", inputs_1.shape)
                print("inputs[2*step][0][:, output_channels :]", inputs[2*step][:, output_channels :].shape, "Slice: ", output_channels, None)
                input_tensor = torch.cat(
                    [
                        inputs_0,
                        inputs_1,
                        inputs[2 * step][:, output_channels :],
                    ],
                    dim=1,
                )   
            else:
                inputs_0 = outputs[-hist-1]
                print("inputs_0", inputs_0.shape, "outputs[]", -hist-1)
                print("inputs[2*step]", inputs[2*step].shape, "Slice: ", output_channels, None)
                input_tensor = torch.cat(
                    [
                        inputs_0,
                        inputs[2 * step][:, output_channels :],
                    ],
                    dim=1,
                )
            
            print(input_tensor.shape)
            
            assert input_tensor.shape[1] == input_channels, f"Input shape is {input_tensor.shape[1]} but should be {input_channels}"
            decodings = forward_once(input_tensor)
            if pred_residuals:
                reshaped = (
                    input_tensor[:, output_channels * hist : output_channels * (hist+1)] + decodings
                )  # Residual prediction
            else:
                reshaped = decodings  # Absolute prediction

            if loss_fn is not None:
                assert reshaped.shape == inputs[2 * step + 1].shape, f"Output shape is {reshaped.shape} but should be {inputs[2 * step + 1].shape}"
                if loss is None:
                    loss = loss_fn(
                        reshaped,
                        inputs[2 * step + 1],
                    )
                else:
                    loss += loss_fn(
                        reshaped,
                        inputs[2 * step + 1],
                    )

            outputs.append(reshaped)

        if loss_fn is None:
            if output_only_last:
                res = outputs[-1]
            else:
                res = outputs
            return res

        else:
            return loss

In [9]:
hist = 1
output_channels = (hist+1)*77
input_channels = (hist+1)*77+3
pred_residuals = False

# HIST=0, 4 steps ; 0->[0in, 1out, 1in, 2out, 2in, 3out, 3in, 4out]
# HIST=1, 4 steps , 0->[[0in, 1in], [2out, 3out], [1in, 2in], [3out, 4out], [2in, 3in], [4out, 5out], [3in, 4in], [5out, 6out]]
# HIST=2, 4 steps , 0->[[0in, 1in, 2in], [3out, 4out, 5out], [1in, 2in, 3in], [4out, 5out, 6out], [2in, 3in, 4in], [5out, 6out, 7out], [3in, 4in, 5in], [6out, 7out, 8out]]

    
train_data = data_CNN_Disk_steps(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_samples,
    lag,
    interval,
    hist,
    steps,
    device="cuda",
)
a = forward(train_data[2], output_only_last=False, loss_fn=torch.nn.MSELoss())

/global/common/software/m4331/suryad/conda/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


0
torch.Size([1, 234, 180, 360])
1
inputs_0 torch.Size([1, 154, 180, 360]) slice:  None 154
inputs_1 torch.Size([1, 77, 180, 360])
inputs[2*step][0][:, output_channels :] torch.Size([1, 3, 180, 360]) Slice:  231 None
torch.Size([1, 234, 180, 360])
2
inputs_0 torch.Size([1, 77, 180, 360]) slice:  None 77
inputs_1 torch.Size([1, 154, 180, 360])
inputs[2*step][0][:, output_channels :] torch.Size([1, 3, 180, 360]) Slice:  231 None
torch.Size([1, 234, 180, 360])
3
inputs_0 torch.Size([1, 231, 180, 360]) outputs[] -3
inputs[2*step] torch.Size([1, 234, 180, 360]) Slice:  231 None
torch.Size([1, 234, 180, 360])


In [38]:
# HIST=0 ; 0->[0, 1]; 1->[2, 3]; 2->[3, 4]; 3->[5, 6]
# HIST=1 ; 0->[[0, 1], [2, 3]]; 1->[[2, 3], [4, 5]]; 2->[[4, 5], [6, 7]]
# HIST=2 ; 0->[[0, 1, 2], [3, 4, 5]]; 1->[[2, 3, 4], [5, 6, 7]];]
def inference(
        inputs, num_steps=None, output_only_last=False, device="cuda"
    ) -> torch.Tensor:
        outputs = []
        for step in range(num_steps):
            print(step)
            if step == 0:
                input_tensor = inputs[0][0].to(device=device) # inputs[0][0] is the input at step 0 
            else:
                inputs_0 = outputs[-1].unsqueeze(0) # Last output corresponding to current input
                print("inputs_0", inputs_0.shape, "outputs[]", -1)
                print("inputs[step][0][:, output_channels :]", inputs[step][0][:, output_channels :].shape, "Slice: ", output_channels, None)
                input_tensor = torch.cat(
                    [
                        inputs_0,
                        inputs[step][0][:, output_channels :] # concatenate the boundary conditions
                        .to(device=device),
                    ],
                    dim=1,
                )
            print(input_tensor.shape)

            assert input_tensor.shape[1] == input_channels, f"Input shape is {input_tensor.shape[1]} but should be {input_channels}"
            decodings = forward_once(input_tensor)
            if pred_residuals:
                reshaped = input_tensor[0, output_channels * hist : output_channels * (hist+1)].to( # Residuals on last state in input
                    device=device
                ) + decodings.squeeze(
                    0
                )
            else:
                reshaped = decodings.squeeze(0)
            
            outputs.append(reshaped)

        if output_only_last:
            res = outputs[-1]
        else:
            res = outputs

        return res
    

In [39]:
hist = 2
s_train = lag * hist
e_train = s_train + N_samples * interval
e_test = e_train + interval * N_val
output_channels = (hist+1)*77
input_channels = (hist+1)*77+3
pred_residuals = False
data = xr.open_zarr(
    os.path.join(data_dir, data_zarr)
)
val_data = data_CNN_Disk(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_val,
    lag,
    interval,
    hist,
    e_train,
    long_rollout=True,
    device="cuda",
)
print(val_data.rolling_indices)
inf = inference(val_data, num_steps=10, output_only_last=False, device="cpu")

/global/common/software/m4331/suryad/conda/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


Long rollout will begin with input and produce output from time index 21 and 23 respectively
<xarray.DataArray (window_dim: 2361, time: 4)>
array([[   0,    1,    2,    3],
       [   2,    3,    4,    5],
       [   4,    5,    6,    7],
       ...,
       [4716, 4717, 4718, 4719],
       [4718, 4719, 4720, 4721],
       [4720, 4721, 4722, 4723]])
Coordinates:
  * time     (time) object 2022-12-14 12:00:00 ... 2022-12-29 12:00:00
Dimensions without coordinates: window_dim
0
torch.Size([1, 157, 180, 360])
1
inputs_0 torch.Size([1, 154, 180, 360]) outputs[] -1
inputs[step][0][:, output_channels :] torch.Size([1, 3, 180, 360]) Slice:  154 None
torch.Size([1, 157, 180, 360])
2
inputs_0 torch.Size([1, 154, 180, 360]) outputs[] -1
inputs[step][0][:, output_channels :] torch.Size([1, 3, 180, 360]) Slice:  154 None
torch.Size([1, 157, 180, 360])
3
inputs_0 torch.Size([1, 154, 180, 360]) outputs[] -1
inputs[step][0][:, output_channels :] torch.Size([1, 3, 180, 360]) Slice:  154 None
torch.Size